In [1]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def create_spectrogram(audio_path, output_path, figsize=(12, 6), dpi=100, freq_low=3400, freq_high=4000):
    """
    Create and save a spectrogram from an audio file (without axes or labels),
    hardcoded to only the freq_low-freq_high Hz band.

    Args:
        audio_path (str): Path to input audio file
        output_path (str): Path to save spectrogram image
        figsize (tuple): Figure size (width, height) in inches
        dpi (int): Dots per inch for output image
        freq_low (float): lower bound of the frequency band to keep (Hz)
        freq_high (float): upper bound of the frequency band to keep (Hz)
    """
    try:
        # Load audio file
        y, sr = librosa.load(audio_path)

        # Perform STFT - n_fft made explicit (was an implicit default before)
        # so it's guaranteed to match the bin lookup used for the band slice below
        n_fft = 2048
        stft = np.abs(librosa.stft(y, n_fft=n_fft))

        # Convert to dB scale
        stft_db = librosa.amplitude_to_db(stft, ref=np.max)

        # ============ NEW: hardcode to the freq_low-freq_high Hz band ============
        freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
        band_mask = (freqs >= freq_low) & (freqs <= freq_high)
        stft_db = stft_db[band_mask, :]
        # ============ END NEW ============

        # Create the spectrogram plot
        plt.figure(figsize=figsize, frameon=False)
        ax = plt.Axes(plt.gcf(), [0., 0., 1., 1.])  # Full figure area
        ax.set_axis_off()  # Turn off all axis decorations
        plt.gcf().add_axes(ax)

        # Display spectrogram without any decorations - now only the cropped band
        librosa.display.specshow(stft_db, sr=sr, cmap='magma')

        # Save the spectrogram (without borders/whitespace)
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0, dpi=dpi)
        plt.close()

    except Exception as e:
        print(f"Error processing {audio_path}: {str(e)}")

def process_dataset(input_root, output_root, freq_low=3400, freq_high=4000):
    """
    Process all WAV files in a dataset to create spectrograms.

    Args:
        input_root (str): Root directory of the audio dataset
        output_root (str): Root directory to save spectrograms (will mirror input structure)
        freq_low (float): lower bound of the frequency band to keep (Hz)
        freq_high (float): upper bound of the frequency band to keep (Hz)
    """
    input_path = Path(input_root)
    output_path = Path(output_root)

    output_path.mkdir(parents=True, exist_ok=True)

    total_files = 0

    for root, dirs, files in os.walk(input_path):
        relative_path = Path(root).relative_to(input_path)
        current_output_dir = output_path / relative_path
        current_output_dir.mkdir(parents=True, exist_ok=True)

        for file in files:
            if file.lower().endswith('.wav'):
                input_file = Path(root) / file
                output_file = current_output_dir / f"{Path(file).stem}.png"

                print(f"Processing: {input_file} -> {output_file}")
                create_spectrogram(str(input_file), str(output_file), freq_low=freq_low, freq_high=freq_high)
                total_files += 1

    print(f"\nFinished processing {total_files} files.")
    print(f"Spectrograms saved to: {output_root}")

if __name__ == "__main__":
    # Configuration
    input_dataset = r"D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split"
    output_dataset = r"D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split"

    process_dataset(input_dataset, output_dataset, freq_low=3400, freq_high=4000)

Processing: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split\test\Bus\Image_11.wav -> D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split\test\Bus\Image_11.png
Processing: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split\test\Bus\Image_12.wav -> D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split\test\Bus\Image_12.png
Processing: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split\test\Bus\Image_19.wav -> D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split\test\Bus\Image_19.png
Processing: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split\test\Bus\Image_20.wav -> D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split\test\Bus\Image_20.png
Processing: D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_split\test\Bus\Image_21.wav -> D:\Documents\Iquisitionis\103v2\data\4Class_Img2Audv5_spectrogram_split\test\Bus\Image_21.png
Processing: D:\Documents\Iquisitionis\103v2\d